# 05 — Phase 4: xG Analysis
**Football Players Stats 2024/25**

Expected Goals deep dive — who beats the model, who underperforms, and how penalty-dependent are top scorers?

> Outfield players with 900+ minutes only.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

player_df = pd.read_csv('Dataset\player_clean.csv')

LEAGUE_COLORS = {
    'Premier League': '#3d195b', 'La Liga': '#ee8707',
    'Serie A': '#008fd7', 'Bundesliga': '#d3010c', 'Ligue 1': '#091c3e',
}
POSITION_COLORS = {'FW': '#e63946', 'MF': '#2a9d8f', 'DF': '#457b9d'}

xg_df = player_df[
    (player_df['sufficient_minutes'] == True) &
    (player_df['primary_position'] != 'GK')
].copy()

print(f"Outfield players with sufficient minutes: {len(xg_df)}")

Outfield players with sufficient minutes: 1457


In [2]:
# 4.1 xG vs Actual Goals — All Outfield Players
fig = px.scatter(xg_df, x='xg', y='goals', color='primary_position',
                 color_discrete_map=POSITION_COLORS, size='minutes_played', size_max=15,
                 hover_data=['player_name', 'club', 'league', 'goals', 'xg', 'goals_vs_xg'],
                 title='xG vs Actual Goals — All Outfield Players (2024/25)',
                 labels={'xg': 'Expected Goals (xG)', 'goals': 'Actual Goals', 'primary_position': 'Position'},
                 opacity=0.7)
max_val = max(xg_df['xg'].max(), xg_df['goals'].max()) + 2
fig.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color='gray', dash='dash', width=1.5))
fig.add_annotation(x=max_val*0.6, y=max_val*0.72, text='Perfect xG line (Goals = xG)',
                   showarrow=False, font=dict(color='gray', size=11), bgcolor='white')
fig.update_layout(height=580, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='Position')
fig.show()

In [3]:
# 4.2 Top 15 xG Overperformers (min 5 xG)
overperformers = xg_df[xg_df['xg'] >= 5].nlargest(15, 'goals_vs_xg').sort_values('goals_vs_xg', ascending=True)

fig = px.bar(overperformers, x='goals_vs_xg', y='player_name', orientation='h',
             color='league', color_discrete_map=LEAGUE_COLORS,
             hover_data=['club', 'league', 'goals', 'xg', 'goals_vs_xg'],
             title='Top 15 xG Overperformers — Goals minus xG (min 5 xG)',
             labels={'goals_vs_xg': 'Goals minus xG', 'player_name': ''}, text='goals_vs_xg')
fig.update_traces(textposition='outside', texttemplate='%{text:.1f}')
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=False), legend_title='League')
fig.show()

In [4]:
# 4.3 Top 15 xG Underperformers (min 5 xG)
underperformers = xg_df[xg_df['xg'] >= 5].nsmallest(15, 'goals_vs_xg').sort_values('goals_vs_xg', ascending=False)

fig = px.bar(underperformers, x='goals_vs_xg', y='player_name', orientation='h',
             color='league', color_discrete_map=LEAGUE_COLORS,
             hover_data=['club', 'league', 'goals', 'xg', 'goals_vs_xg'],
             title='Top 15 xG Underperformers — Goals minus xG (min 5 xG)',
             labels={'goals_vs_xg': 'Goals minus xG', 'player_name': ''}, text='goals_vs_xg')
fig.update_traces(textposition='outside', texttemplate='%{text:.1f}')
fig.update_layout(height=520, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=False), legend_title='League')
fig.show()

In [5]:
# 4.4 Average xG vs Goals by League — Forwards Only
fw_only = xg_df[xg_df['primary_position'] == 'FW'].copy()
league_xg = fw_only.groupby('league').agg(
    avg_xg=('xg', 'mean'), avg_goals=('goals', 'mean'),
    avg_goals_vs_xg=('goals_vs_xg', 'mean'), player_count=('player_name', 'count')
).reset_index().round(2)

fig = go.Figure()
for _, row in league_xg.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['avg_xg']], y=[row['avg_goals']], mode='markers+text',
        name=row['league'], marker=dict(color=LEAGUE_COLORS[row['league']],
        size=row['player_count']/3, line=dict(width=1, color='white')),
        text=[row['league']], textposition='top center',
        hovertemplate=f"<b>{row['league']}</b><br>Avg xG: {row['avg_xg']}<br>Avg Goals: {row['avg_goals']}"
    ))
max_val = max(league_xg['avg_xg'].max(), league_xg['avg_goals'].max()) + 0.5
fig.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color='gray', dash='dash', width=1.5))
fig.update_layout(title='Avg xG vs Avg Goals by League — Forwards Only (2024/25)',
                  xaxis_title='Average xG per Forward', yaxis_title='Average Goals per Forward',
                  height=500, plot_bgcolor='white', showlegend=False,
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [6]:
# 4.5 xG Distribution by Position
fig = px.box(xg_df, x='primary_position', y='xg', color='primary_position',
             color_discrete_map=POSITION_COLORS,
             title='xG Distribution by Position (2024/25)',
             labels={'xg': 'Expected Goals (xG)', 'primary_position': 'Position'},
             points='outliers', category_orders={'primary_position': ['FW', 'MF', 'DF']})
fig.update_layout(height=480, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [7]:
# 4.6 npxG vs xG — Penalty Dependency of Top 20 Forwards
top_fw = xg_df[xg_df['primary_position'] == 'FW'].nlargest(20, 'xg').copy()
top_fw = top_fw.sort_values('xg', ascending=True)
top_fw['penalty_xg'] = (top_fw['xg'] - top_fw['npxg']).round(2)

fig = go.Figure()
fig.add_trace(go.Bar(name='Non-Penalty xG', y=top_fw['player_name'],
                     x=top_fw['npxg'], orientation='h', marker_color='#e63946',
                     text=top_fw['npxg'].round(1), textposition='inside'))
fig.add_trace(go.Bar(name='Penalty xG', y=top_fw['player_name'],
                     x=top_fw['penalty_xg'], orientation='h', marker_color='#f4a261',
                     text=top_fw['penalty_xg'].round(1), textposition='inside'))
fig.update_layout(barmode='stack',
                  title='Top 20 Forwards — xG Breakdown: Non-Penalty vs Penalty (2024/25)',
                  xaxis_title='Expected Goals (xG)', height=580, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='xG Type')
fig.show()